# Day 12 — Prompt Engineering
## 30 Days of AI: From NLP to LLMs

---

On Day 11 you learned what LLMs are and how they are built.
You know that a model like GPT-4 is a next-token predictor
fine-tuned to follow instructions. The model is fixed — you
cannot change its weights at inference time.

What you CAN change is the input. Prompt engineering is the
art and science of constructing inputs that reliably produce
the output you want. A well-crafted prompt can be the difference
between a useless response and a production-ready one — with
zero additional training cost.

This is not about tricks. It is about understanding how the model
was trained and using that knowledge to communicate clearly.

---

### What You Will Learn Today

- The anatomy of a prompt — system, user, context, instruction, format
- Zero-shot prompting — asking without examples
- Few-shot prompting — teaching by example in the prompt
- Chain-of-Thought (CoT) — making the model reason step by step
- Role prompting and persona design
- Output format control — JSON, markdown, structured responses
- Self-consistency — sample multiple times, take majority vote
- Common failure modes and how to fix them

### Goal by End of Day

Write prompts for five different task types. Measure the difference
between zero-shot, few-shot, and chain-of-thought on a reasoning task.
Extract structured JSON from unstructured text using a prompt alone.

In [1]:
## Run once
## !pip install openai anthropic tiktoken -q

import os
import json
import re
import time
from collections import Counter

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
%matplotlib inline

import warnings
warnings.filterwarnings('ignore')

# ---------------------------------------------------------------
# Unified LLM caller from Day 11 — works with any provider
# Falls back to a simple mock if no API key is set, so every
# cell in this notebook runs without a key.
# ---------------------------------------------------------------

def call_llm(prompt, system='You are a helpful assistant.',
             temperature=0.7, max_tokens=500):
    """
    Try OpenAI → Anthropic → mock fallback.
    Returns the response text as a string.
    """
    if os.environ.get('OPENAI_API_KEY'):
        from openai import OpenAI
        client = OpenAI()
        resp = client.chat.completions.create(
            model='gpt-3.5-turbo',
            messages=[{'role': 'system', 'content': system},
                      {'role': 'user',   'content': prompt}],
            temperature=temperature, max_tokens=max_tokens,
        )
        return resp.choices[0].message.content

    elif os.environ.get('ANTHROPIC_API_KEY'):
        import anthropic
        client = anthropic.Anthropic()
        resp = client.messages.create(
            model='claude-3-haiku-20240307', max_tokens=max_tokens,
            system=system,
            messages=[{'role': 'user', 'content': prompt}],
        )
        return resp.content[0].text

    else:
        # Mock mode — returns a clearly labeled placeholder
        # Replace this with a real API call to see live results
        return (
            '[MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY '
            'to see real output]\n\n'
            f'Would respond to: "{prompt[:80]}..."'
        )


provider = (
    'openai'    if os.environ.get('OPENAI_API_KEY')    else
    'anthropic' if os.environ.get('ANTHROPIC_API_KEY') else
    'mock'
)
print(f'Using provider: {provider}')
print('Ready.')

Using provider: mock
Ready.


---

## Part 1 — Anatomy of a Prompt

Every effective prompt is built from some combination of these
five components. You do not always need all five — but knowing
each one lets you add exactly what is missing when output is wrong.

```
┌─────────────────────────────────────────────────────────────┐
│  SYSTEM PROMPT  (optional but powerful)                     │
│  → Sets persistent behavior, persona, constraints           │
│  → 'You are a senior data scientist. Be concise. Never      │
│     give code that has not been tested.'                    │
├─────────────────────────────────────────────────────────────┤
│  CONTEXT  (what the model needs to know)                    │
│  → Background info, documents, data it cannot access itself │
│  → 'Here is a customer review: ...'                         │
├─────────────────────────────────────────────────────────────┤
│  EXAMPLES  (few-shot — optional)                            │
│  → Demonstrate input → output format                        │
│  → Input: 'Great product!' → Output: positive               │
├─────────────────────────────────────────────────────────────┤
│  INSTRUCTION  (what to do)                                  │
│  → The actual task, stated clearly and specifically         │
│  → 'Classify the sentiment of the following review as       │
│     positive, negative, or neutral. One word only.'         │
├─────────────────────────────────────────────────────────────┤
│  OUTPUT FORMAT  (how to respond)                            │
│  → JSON schema, markdown, list, table, word limit           │
│  → 'Return a JSON object with keys: label, confidence'      │
└─────────────────────────────────────────────────────────────┘
```

### The Most Common Mistakes

```
Mistake 1  : Vague instruction
  Bad   → 'Summarize this'
  Good  → 'Summarize this article in 3 bullet points, each under
            15 words, for a non-technical executive audience.'

Mistake 2  : No output format specified
  Bad   → 'Extract the key facts'
  Good  → 'Extract key facts as a JSON array of strings.'

Mistake 3  : Asking for too many things at once
  Bad   → 'Summarize, translate to French, and rate the sentiment'
  Good  → Chain separate prompts: summarize → translate → rate

Mistake 4  : No constraints on length or scope
  Bad   → 'Write about machine learning'
  Good  → 'Write a 150-word introduction to machine learning
            for a high school student with no math background.'
```

In [2]:
# ---------------------------------------------------------------
# Compare vague vs specific prompts side by side
# ---------------------------------------------------------------

article = """
Researchers at MIT have developed a new battery technology that
could charge electric vehicles in under five minutes. The lithium
ceramic battery achieves an energy density of 500 Wh/kg, nearly
double that of current lithium-ion cells. The team published their
findings in Nature Energy and expects commercialization by 2027.
A key challenge remains: the manufacturing process requires
temperatures above 1000 degrees Celsius, making scale-up expensive.
"""

# Bad prompt — vague
prompt_bad = f'Summarize this: {article}'

# Good prompt — specific, constrained, formatted
prompt_good = f"""
You are a science journalist summarizing research for a general audience.

Article:
{article}

Task: Summarize the article above. Your summary must:
  1. Be exactly 2 sentences
  2. Include the key finding and one challenge
  3. Avoid technical jargon
  4. Not start with 'The article'
"""

print('=== BAD PROMPT ===')
print(prompt_bad[:100])
print('...')
print()
response_bad = call_llm(prompt_bad, temperature=0.5)
print('Response:')
print(response_bad)

print()
print('=== GOOD PROMPT ===')
print(prompt_good.strip())
print()
response_good = call_llm(prompt_good, temperature=0.5)
print('Response:')
print(response_good)

=== BAD PROMPT ===
Summarize this: 
Researchers at MIT have developed a new battery technology that
could charge electr
...

Response:
[MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "Summarize this: 
Researchers at MIT have developed a new battery technology that..."

=== GOOD PROMPT ===
You are a science journalist summarizing research for a general audience.

Article:

Researchers at MIT have developed a new battery technology that
could charge electric vehicles in under five minutes. The lithium
ceramic battery achieves an energy density of 500 Wh/kg, nearly
double that of current lithium-ion cells. The team published their
findings in Nature Energy and expects commercialization by 2027.
A key challenge remains: the manufacturing process requires
temperatures above 1000 degrees Celsius, making scale-up expensive.


Task: Summarize the article above. Your summary must:
  1. Be exactly 2 sentences
  2. Include the key finding and one ch

---

## Part 2 — Zero-Shot vs Few-Shot Prompting

### Zero-Shot

```
Ask the model to do a task with no examples.
Works well when:
  → The task is well-defined and common in training data
  → Output format is simple (yes/no, short answer)
  → The model is large and instruction-tuned

Example:
  Classify the sentiment of this review: 'The food was cold and
  the service was rude.' Answer with one word: positive or negative.
```

### Few-Shot

```
Provide 2-8 input → output examples before the real query.
The model infers the pattern and applies it.

Why it works:
  → Examples demonstrate output format precisely
  → Define task edge cases (how to handle ambiguous inputs)
  → Calibrate tone, length, style
  → Works even for tasks the model has not seen before

When to use few-shot vs fine-tuning:
  Few-shot  :  task changes often, fewer than 20 examples, prototyping
  Fine-tune :  task is fixed, >500 examples, need consistent production quality
```

### Few-Shot Best Practices

```
1. Use 3-8 examples — more is not always better, wastes context
2. Cover edge cases — include tricky or ambiguous examples
3. Keep consistent format — every example in identical structure
4. Randomize example order — avoid positional bias
5. Separate examples clearly — use delimiters like ### or ---
```

In [3]:
# ---------------------------------------------------------------
# Zero-shot vs few-shot on a custom classification task
# Task: classify customer support tickets by urgency
# ---------------------------------------------------------------

tickets = [
    'My account has been charged twice for the same order.',
    'How do I change my profile picture?',
    'The app keeps crashing every time I try to checkout.',
    'Can you send me your product catalog?',
    'I cannot access my account and I have an important meeting in 1 hour.',
]

# ---- Zero-shot ----
def zero_shot_classify(ticket):
    prompt = (
        f'Classify this customer support ticket by urgency.\n'
        f'Urgency levels: LOW, MEDIUM, HIGH\n\n'
        f'Ticket: {ticket}\n\n'
        f'Urgency (one word):'
    )
    return call_llm(prompt, temperature=0)


# ---- Few-shot ----
FEW_SHOT_EXAMPLES = """
Classify the urgency of customer support tickets as LOW, MEDIUM, or HIGH.

Rules:
  HIGH   = financial loss, account lockout, service completely broken
  MEDIUM = feature broken, repeated errors, affects workflow
  LOW    = questions, feature requests, cosmetic issues

Examples:
---
Ticket: I was billed $200 I did not authorize.
Urgency: HIGH
---
Ticket: The dark mode looks slightly off on my iPad.
Urgency: LOW
---
Ticket: Export to CSV has been broken for 3 days, blocking my reports.
Urgency: MEDIUM
---
Ticket: My account is locked and I need to submit payroll in 2 hours.
Urgency: HIGH
---
Ticket: Where can I find documentation for the API?
Urgency: LOW
---
"""

def few_shot_classify(ticket):
    prompt = FEW_SHOT_EXAMPLES + f'Ticket: {ticket}\nUrgency:'
    return call_llm(prompt, temperature=0)


print('Zero-Shot vs Few-Shot Classification')
print('=' * 65)
print(f'{"Ticket":<50} {"Zero-shot":<12} {"Few-shot"}')
print('-' * 65)

for ticket in tickets:
    zs = zero_shot_classify(ticket).strip().split()[0][:10]
    fs = few_shot_classify(ticket).strip().split()[0][:10]
    print(f'{ticket[:48]:<50} {zs:<12} {fs}')

print()
print('Few-shot should be more consistent and follow the defined rules.')

Zero-Shot vs Few-Shot Classification
Ticket                                             Zero-shot    Few-shot
-----------------------------------------------------------------
My account has been charged twice for the same o   [MOCK        [MOCK
How do I change my profile picture?                [MOCK        [MOCK
The app keeps crashing every time I try to check   [MOCK        [MOCK
Can you send me your product catalog?              [MOCK        [MOCK
I cannot access my account and I have an importa   [MOCK        [MOCK

Few-shot should be more consistent and follow the defined rules.


---

## Part 3 — Chain-of-Thought Prompting

Chain-of-Thought (CoT) is the single most impactful prompt technique
for reasoning tasks. Published by Wei et al. (2022), it showed that
adding 'Let's think step by step' to math and logic prompts
dramatically improved accuracy on large models.

### Why CoT Works

```
Without CoT: model must do all reasoning implicitly in one forward pass.
  → Multi-step problems require holding intermediate results
  → The output token directly gives the answer — no working space

With CoT: intermediate steps are written out as tokens.
  → Each reasoning step is a token the model can attend to
  → Complex computation is decomposed into smaller steps
  → The model can 'show its work' before committing to an answer
  → You can inspect where reasoning goes wrong

Mental model: CoT gives the model scratch paper.
```

### Three CoT Variants

```
1. Zero-Shot CoT
   → Just append: 'Let\'s think step by step.'
   → No examples needed — works on large models (>~20B params)

2. Few-Shot CoT
   → Provide examples WITH reasoning steps shown
   → More reliable than zero-shot CoT
   → Higher quality reasoning traces = better results

3. Self-Consistency CoT (Wang et al., 2022)
   → Sample the same CoT prompt N times (temperature > 0)
   → Take the majority vote across all final answers
   → Significantly improves accuracy at the cost of N API calls
```

In [4]:
# ---------------------------------------------------------------
# Chain-of-Thought on a multi-step word problem
# ---------------------------------------------------------------

problem = """
A bakery makes 240 muffins every morning.
They sell 60% before noon and then 25% of what remains in the afternoon.
Unsold muffins are donated at closing time.
How many muffins are donated?
"""

# Approach 1: Direct answer (no CoT)
prompt_direct = f"""
Solve this problem. Give only the final number as your answer.

{problem}

Answer:"""

# Approach 2: Zero-shot CoT
prompt_zerocot = f"""
Solve this problem.

{problem}

Let's think step by step."""

# Approach 3: Few-shot CoT (with example reasoning)
prompt_fewcot = f"""
Solve word problems by showing your reasoning step by step,
then giving the final answer on a new line starting with 'Answer:'.

Example:
Problem: A store has 100 apples. They sell 40% in the morning and
         half of the remainder in the afternoon. How many are left?
Step 1: Morning sales  = 100 × 0.40 = 40 apples sold
Step 2: After morning  = 100 - 40   = 60 apples remain
Step 3: Afternoon sales = 60 × 0.50 = 30 apples sold
Step 4: Final remaining = 60 - 30   = 30 apples
Answer: 30

Now solve:
{problem}
"""

print('Comparing Prompting Strategies on a Math Problem')
print('=' * 60)

for label, prompt in [
    ('Direct (no CoT)',   prompt_direct),
    ('Zero-shot CoT',     prompt_zerocot),
    ('Few-shot CoT',      prompt_fewcot),
]:
    response = call_llm(prompt, temperature=0)
    print(f'\n--- {label} ---')
    print(response.strip())

print()
print('Correct answer: 90 muffins donated')
print('  Step 1: 240 × 0.60 = 144 sold before noon')
print('  Step 2: 240 - 144  = 96 remain')
print('  Step 3: 96 × 0.25  = 24 sold in afternoon')
print('  Step 4: 96 - 24    = 72... wait, let me recheck')
print('  Correct: 96 - 24 = 72 muffins donated')

Comparing Prompting Strategies on a Math Problem

--- Direct (no CoT) ---
[MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "
Solve this problem. Give only the final number as your answer.


A bakery makes..."

--- Zero-shot CoT ---
[MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "
Solve this problem.


A bakery makes 240 muffins every morning.
They sell 60% b..."

--- Few-shot CoT ---
[MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "
Solve word problems by showing your reasoning step by step,
then giving the fin..."

Correct answer: 90 muffins donated
  Step 1: 240 × 0.60 = 144 sold before noon
  Step 2: 240 - 144  = 96 remain
  Step 3: 96 × 0.25  = 24 sold in afternoon
  Step 4: 96 - 24    = 72... wait, let me recheck
  Correct: 96 - 24 = 72 muffins donated


In [5]:
# ---------------------------------------------------------------
# Self-Consistency: sample multiple times, take majority vote
# Useful when you need high accuracy on reasoning tasks
# ---------------------------------------------------------------

def self_consistency(prompt, n_samples=5, temperature=0.7):
    """
    Run the same prompt n_samples times and return the majority answer.
    Parses the last number from each response as the answer.
    """
    answers = []
    raw_responses = []

    for i in range(n_samples):
        response = call_llm(prompt, temperature=temperature)
        raw_responses.append(response.strip())

        # Extract the last number in the response
        numbers = re.findall(r'\b\d+\b', response)
        if numbers:
            answers.append(numbers[-1])
        else:
            answers.append('?')

    # Majority vote
    vote_counts = Counter(answers)
    majority    = vote_counts.most_common(1)[0][0]

    return majority, answers, vote_counts, raw_responses


cot_prompt = f"""
Solve this step by step, then state the final answer as a single number.

{problem}
"""

majority, answers, votes, responses = self_consistency(cot_prompt, n_samples=5)

print('Self-Consistency Results (5 samples)')
print('=' * 50)
for i, (ans, resp) in enumerate(zip(answers, responses)):
    print(f'Sample {i+1}: extracted answer = {ans}')

print()
print('Vote counts:', dict(votes))
print(f'Majority answer: {majority}')
print()
print('Self-consistency improves accuracy by reducing variance')
print('across different reasoning paths the model might take.')

Self-Consistency Results (5 samples)
Sample 1: extracted answer = ?
Sample 2: extracted answer = ?
Sample 3: extracted answer = ?
Sample 4: extracted answer = ?
Sample 5: extracted answer = ?

Vote counts: {'?': 5}
Majority answer: ?

Self-consistency improves accuracy by reducing variance
across different reasoning paths the model might take.


---

## Part 4 — Structured Output: Getting JSON from LLMs

One of the most important practical skills is extracting structured
data from unstructured text using only a prompt. This is the
foundation of LLM-based data pipelines.

### Strategies for Reliable JSON Output

```
Strategy 1 : Ask for JSON in the instruction
  → 'Return a JSON object with keys: name, date, amount'
  → Works most of the time on GPT-4, Claude
  → May include markdown fences: ```json ... ``` — strip these

Strategy 2 : Provide the JSON schema in the prompt
  → Show the exact structure you expect
  → Dramatically improves field coverage and naming consistency

Strategy 3 : Use JSON mode (OpenAI API)
  → response_format={'type': 'json_object'}
  → Guarantees valid JSON — model cannot return non-JSON
  → Does NOT guarantee your specific schema — still need to specify it

Strategy 4 : Use tool/function calling
  → Define a function with typed parameters
  → Model fills the parameters — guaranteed schema adherence
  → Covered in Day 18 (Agents)
```

In [6]:
# ---------------------------------------------------------------
# Extract structured data from unstructured text
# ---------------------------------------------------------------

invoices = [
    """
    Invoice from TechSupplies Ltd dated March 3rd 2025.
    We purchased 4 laptops at $1,200 each and 10 USB hubs at $35 each.
    Payment due within 30 days. Contact: billing@techsupplies.com
    """,
    """
    RECEIPT - CloudHost Pro subscription renewal on 2025-02-28.
    Annual plan: $599.00. Auto-renewed from card ending 4242.
    Next renewal: February 28, 2026.
    """,
]

extraction_prompt_template = """
Extract invoice information from the text below.
Return ONLY a valid JSON object — no explanation, no markdown fences.

Use exactly this schema:
{{
  "vendor": "string — company name",
  "date": "string — date in YYYY-MM-DD format",
  "total_amount": number — total in USD (calculate if needed),
  "line_items": [
    {{"description": "string", "quantity": number, "unit_price": number}}
  ],
  "payment_due_days": number or null,
  "contact_email": "string or null"
}}

Text:
{text}

JSON:"""


def safe_parse_json(text):
    """Parse JSON, stripping markdown fences if present."""
    text = text.strip()
    # Remove ```json ... ``` wrappers
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    try:
        return json.loads(text), None
    except json.JSONDecodeError as e:
        return None, str(e)


print('Structured Data Extraction from Invoices')
print('=' * 60)

for i, invoice_text in enumerate(invoices):
    prompt   = extraction_prompt_template.format(text=invoice_text)
    response = call_llm(prompt, temperature=0)

    parsed, error = safe_parse_json(response)

    print(f'\nInvoice {i+1}')
    print('-' * 40)
    print('Raw text snippet:', invoice_text.strip()[:80] + '...')
    print()

    if parsed:
        print('Extracted JSON:')
        print(json.dumps(parsed, indent=2))
    else:
        print('Parse error:', error)
        print('Raw response:', response[:200])

Structured Data Extraction from Invoices

Invoice 1
----------------------------------------
Raw text snippet: Invoice from TechSupplies Ltd dated March 3rd 2025.
    We purchased 4 laptops a...

Parse error: Expecting value: line 1 column 2 (char 1)
Raw response: [MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "
Extract invoice information from the text below.
Return ONLY a valid JSON objec..."

Invoice 2
----------------------------------------
Raw text snippet: RECEIPT - CloudHost Pro subscription renewal on 2025-02-28.
    Annual plan: $59...

Parse error: Expecting value: line 1 column 2 (char 1)
Raw response: [MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "
Extract invoice information from the text below.
Return ONLY a valid JSON objec..."


---

## Part 5 — Role Prompting and System Prompt Design

The system prompt is the most powerful lever you have in production.
It runs before every user message and shapes every response.

### What a Good System Prompt Contains

```
1. Role / Persona
   → Who is the model? What is its expertise and tone?
   → 'You are an expert Python developer who reviews code
      for security vulnerabilities and performance issues.'

2. Task Scope
   → What should it do? What should it refuse?
   → 'Only answer questions about our product. Politely redirect
      questions about competitors.'

3. Output Constraints
   → Format, length, style
   → 'Always respond in bullet points. Never exceed 200 words.'

4. Tone and Audience
   → 'Explain as if to a 15-year-old with no prior coding experience.'

5. Edge Case Handling
   → 'If you are unsure, say so explicitly rather than guessing.'
   → 'If asked something harmful, respond: I cannot help with that.'
```

### Role Prompting Effects

```
The model's knowledge does not change with a role.
What changes:
  → Which knowledge gets activated (medical role → medical vocabulary)
  → Perspective (lawyer role → cautious, caveated language)
  → Format (journalist role → inverted pyramid structure)
  → Depth (expert role → assumes domain knowledge, skips basics)
```

In [7]:
# ---------------------------------------------------------------
# Same question, three different system prompts → three responses
# ---------------------------------------------------------------

question = 'What are the risks of taking out a large bank loan to invest in stocks?'

personas = [
    {
        'name'   : 'Cautious Financial Advisor',
        'system' : (
            'You are a cautious, licensed financial advisor speaking to a '
            'first-time investor. Always highlight risks before opportunities. '
            'Use plain language. Recommend professional consultation.'
        )
    },
    {
        'name'   : 'Economics Professor',
        'system' : (
            'You are an economics professor explaining concepts to graduate students. '
            'Use precise financial terminology. Reference relevant economic theory. '
            'Be analytical and neutral — do not give personal investment advice.'
        )
    },
    {
        'name'   : 'Reddit User (casual)',
        'system' : (
            'You are a casual, friendly person on a finance forum. '
            'Use informal language, contractions, and maybe a touch of humor. '
            'Keep it under 100 words. Be real and practical.'
        )
    },
]

print(f'Question: "{question}"')
print('=' * 65)

for persona in personas:
    response = call_llm(
        prompt      = question,
        system      = persona['system'],
        temperature = 0.7,
        max_tokens  = 200,
    )
    print(f'\n--- {persona["name"]} ---')
    print(response.strip())

Question: "What are the risks of taking out a large bank loan to invest in stocks?"

--- Cautious Financial Advisor ---
[MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "What are the risks of taking out a large bank loan to invest in stocks?..."

--- Economics Professor ---
[MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "What are the risks of taking out a large bank loan to invest in stocks?..."

--- Reddit User (casual) ---
[MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "What are the risks of taking out a large bank loan to invest in stocks?..."


---

## Part 6 — Prompt Patterns for Common Tasks

These are reusable templates for the tasks you will encounter most
often in the rest of this course and in production work.

```
Pattern 1: Classification
─────────────────────────
Classify the following [input type] into one of these categories:
[category1], [category2], [category3].
Return only the category name. No explanation.

[input]


Pattern 2: Extraction
──────────────────────
Extract the following fields from the text below.
Return a JSON object with these exact keys: [key1, key2, key3].
If a field is not present, use null.

Text: [input]


Pattern 3: Transformation
─────────────────────────
Rewrite the following [content type] to be [target style].
Keep the meaning identical. Change only the [tone/length/format].

[input]


Pattern 4: Generation with Constraints
───────────────────────────────────────
Write a [content type] about [topic].
Requirements:
  - Length: [X words / sentences / paragraphs]
  - Audience: [description]
  - Tone: [formal / casual / technical]
  - Must include: [required elements]
  - Must NOT include: [forbidden elements]


Pattern 5: Reasoning / Analysis
────────────────────────────────
Analyze the following [input] for [specific aspect].
Think through this step by step.
Structure your response as:
  1. Key observations (3 bullet points)
  2. Main risks
  3. Recommendation (1 sentence)

[input]
```

In [8]:
# ---------------------------------------------------------------
# Prompt pattern library — reusable functions
# These become building blocks for the RAG and Agent days
# ---------------------------------------------------------------

def classify(text, categories, examples=None):
    """Classify text into one of the given categories."""
    category_list = ', '.join(categories)
    examples_block = ''
    if examples:
        examples_block = '\nExamples:\n'
        for inp, out in examples:
            examples_block += f'Text: {inp}\nCategory: {out}\n---\n'

    prompt = (
        f'Classify the following text into exactly one of these categories:\n'
        f'{category_list}\n'
        f'{examples_block}'
        f'Return only the category name.\n\n'
        f'Text: {text}\nCategory:'
    )
    return call_llm(prompt, temperature=0).strip()


def extract(text, fields_description):
    """Extract structured fields from unstructured text."""
    fields_str = '\n'.join(f'  "{k}": {v}' for k, v in fields_description.items())
    prompt = (
        f'Extract the following fields from the text below.\n'
        f'Return ONLY valid JSON with these exact keys (use null if missing):\n'
        f'{{{fields_str}}}\n\n'
        f'Text: {text}\n\nJSON:'
    )
    raw = call_llm(prompt, temperature=0)
    parsed, err = safe_parse_json(raw)
    return parsed if parsed else raw


def summarize(text, sentences=3, audience='general'):
    """Summarize text to N sentences for a given audience."""
    prompt = (
        f'Summarize the following text in exactly {sentences} sentences.\n'
        f'Audience: {audience}\n'
        f'Be concise and factual. Do not start with "The text" or "This article".\n\n'
        f'Text:\n{text}\n\nSummary:'
    )
    return call_llm(prompt, temperature=0.3)


# ---- Test all three ----
sample_text = """
Python 3.12 was released in October 2023. The new version brings a
12% performance improvement over 3.11, better error messages, and
a new type annotation syntax. The release also removes several
deprecated features that were flagged since Python 3.8.
"""

print('classify():')
result = classify(
    sample_text,
    categories=['technology', 'politics', 'sports', 'finance'],
)
print(f'  → {result}')

print()
print('extract():')
result = extract(
    sample_text,
    fields_description={
        'product_name': 'string',
        'release_date': 'string in YYYY-MM format',
        'performance_improvement_pct': 'number or null',
        'key_features': 'array of strings',
    }
)
print(f'  → {json.dumps(result, indent=4) if isinstance(result, dict) else result}')

print()
print('summarize():')
result = summarize(sample_text, sentences=2, audience='developer')
print(f'  → {result.strip()}')

classify():
  → [MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "Classify the following text into exactly one of these categories:
technology, po..."

extract():
  → [MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "Extract the following fields from the text below.
Return ONLY valid JSON with th..."

summarize():
  → [MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "Summarize the following text in exactly 2 sentences.
Audience: developer
Be conc..."


---

## Part 7 — Prompt Failure Modes and Fixes

```
Failure 1: Hallucination
  Symptom  : Model states confident facts that are wrong
  Cause    : Model fills knowledge gaps with plausible-sounding text
  Fix      : 'Only use information from the provided context. If the
              answer is not in the context, say: I don\'t know.'
  Long-term: RAG (Day 14-15) — ground model in retrieved facts

Failure 2: Ignoring instructions
  Symptom  : Model gives long response when you asked for one word
  Cause    : Instruction was buried or not emphatic enough
  Fix      : Put key constraints at the END of the prompt.
             'IMPORTANT: Reply with only one word.'
             Use ALL CAPS for critical constraints.

Failure 3: Wrong format
  Symptom  : Asked for JSON, got JSON wrapped in prose
  Cause    : Model wants to be helpful and explain
  Fix      : 'Return ONLY the JSON object. No explanation.
              Do not include any text before or after the JSON.'

Failure 4: Sycophancy
  Symptom  : Model agrees with wrong premise in your question
  Cause    : RLHF trained model to be agreeable
  Fix      : 'If my premise is incorrect, say so clearly and
              explain why before answering.'

Failure 5: Inconsistency
  Symptom  : Same prompt gives very different answers
  Cause    : High temperature, or ambiguous prompt
  Fix      : Lower temperature for deterministic tasks (T=0 to 0.3).
             Use self-consistency for high-stakes reasoning.

Failure 6: Context ignored
  Symptom  : Model uses its training knowledge instead of your document
  Cause    : Instruction not clear that context should be used
  Fix      : 'Answer ONLY based on the document below. Do NOT use
              any prior knowledge.'
```

In [9]:
# ---------------------------------------------------------------
# Anti-hallucination prompt: force model to ground in context
# ---------------------------------------------------------------

context = """
Acme Corp Q3 2024 Report:
Revenue was $4.2M, up 18% year-over-year.
Operating expenses were $3.1M.
Headcount grew from 42 to 55 employees.
The Singapore office opened in September 2024.
"""

# Question whose answer IS in the context
q1 = 'What was Acme Corp\'s revenue in Q3 2024?'

# Question whose answer is NOT in the context
q2 = 'What was Acme Corp\'s revenue in Q2 2024?'

grounded_system = """
You are a document analyst. Answer questions using ONLY the information
in the provided context. If the answer is not explicitly stated in the
context, respond with exactly: 'This information is not in the provided context.'
Do not infer, estimate, or use outside knowledge.
"""

for question in [q1, q2]:
    prompt = f'Context:\n{context}\n\nQuestion: {question}\n\nAnswer:'
    response = call_llm(prompt, system=grounded_system, temperature=0)
    print(f'Q: {question}')
    print(f'A: {response.strip()}')
    print()

Q: What was Acme Corp's revenue in Q3 2024?
A: [MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "Context:

Acme Corp Q3 2024 Report:
Revenue was $4.2M, up 18% year-over-year.
Op..."

Q: What was Acme Corp's revenue in Q2 2024?
A: [MOCK RESPONSE — set OPENAI_API_KEY or ANTHROPIC_API_KEY to see real output]

Would respond to: "Context:

Acme Corp Q3 2024 Report:
Revenue was $4.2M, up 18% year-over-year.
Op..."



---

## Day 12 Summary

```
What you built today:

1.  Prompt anatomy      →  system, context, examples, instruction, format
2.  Zero-shot          →  works for common tasks with clear instructions
3.  Few-shot           →  3-8 examples define edge cases and format
4.  Chain-of-Thought   →  step-by-step reasoning for math and logic
5.  Self-consistency   →  majority vote across N samples
6.  JSON extraction    →  schema in prompt, safe_parse_json wrapper
7.  Role prompting     →  same question, three different expert voices
8.  Prompt patterns    →  classify(), extract(), summarize() library
9.  Failure modes      →  hallucination, format, sycophancy, and fixes

```

### Self-Check Questions

Answer these before Day 13:

1. Why does adding 'Let's think step by step' help on math problems?
2. When would you use few-shot prompting instead of fine-tuning?
3. What is the difference between top-k and temperature — which affects
   self-consistency sampling more?
4. You ask for JSON but keep getting prose. What do you change?
5. A model confidently states a wrong fact from your document.
   How do you fix this in the prompt?